# Tennis Tracking Overlay Demo

This notebook renders the **full selected video** with court, player, and ball overlays and displays the resulting MP4 directly in the notebook.

It uses a **fast overlay profile with accelerator preference**:
- prefers CUDA when available
- otherwise uses MPS when available
- requires an accelerator and does not use CPU for this notebook path
- shows live progress, elapsed time, and ETA while rendering
- writes only overlay video + JSON for this demo path

All cached outputs are written directly into `out/` with a per-video filename.


In [1]:
from __future__ import annotations

import json
import sys
import threading
import traceback
from pathlib import Path
from time import perf_counter
from urllib.parse import quote

import ipywidgets as widgets
from IPython.display import display

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tennis_tracker.events import slugify_video_name
from tennis_tracker.pipeline import run_pipeline
from tennis_tracker.runtime import detect_runtime

SAMPLES_DIR = ROOT / "data" / "samples"
RAW_DIR = ROOT / "data" / "raw"
SAMPLE_VIDEOS = sorted(SAMPLES_DIR.glob("*.mp4"))
RAW_VIDEOS = sorted(RAW_DIR.glob("*.mp4"))
VIDEO_OPTIONS = SAMPLE_VIDEOS + RAW_VIDEOS
assert VIDEO_OPTIONS, "No videos found under data/samples or data/raw."

DEFAULT_VIDEO = next((p for p in VIDEO_OPTIONS if p.name == "broadcast_8s.mp4"), VIDEO_OPTIONS[0])
WEIGHTS_PATH = ROOT / "models" / "tracknet_weights.pth"
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(exist_ok=True)

assert WEIGHTS_PATH.exists(), f"Missing TrackNet weights: {WEIGHTS_PATH}"
runtime = detect_runtime()

def overlay_profile(runtime_name: str) -> dict[str, object]:
    if runtime_name == "cuda":
        return {
            "label": "cuda_fast",
            "player_detect_stride": 2,
            "court_detect_stride": 2,
            "ball_detect_stride": 2,
            "player_model_imgsz": 640,
            "ball_batch_size": 24,
            "write_xml": False,
            "write_events": False,
            "require_cuda": True,
        }
    if runtime_name == "mps":
        return {
            "label": "mps_fast",
            "player_detect_stride": 2,
            "court_detect_stride": 2,
            "ball_detect_stride": 2,
            "player_model_imgsz": 576,
            "ball_batch_size": 18,
            "write_xml": False,
            "write_events": False,
            "require_cuda": False,
        }
    raise RuntimeError("This overlay notebook requires CUDA or MPS. CPU fallback is disabled.")

FAST_PROFILE = overlay_profile(runtime.torch_device)
PIPELINE_PROFILE = {key: value for key, value in FAST_PROFILE.items() if key != "label"}
PROGRESS_STAGE_WEIGHTS = {
    "load_frames": (0.0, 0.10),
    "ball_inference": (0.10, 0.58),
    "compose_overlay": (0.68, 0.28),
    "finalize_video": (0.96, 0.04),
    "done": (1.0, 0.0),
}

def display_label(video_path: Path) -> str:
    if RAW_DIR in video_path.parents:
        return f"raw/{video_path.name}"
    return f"samples/{video_path.name}"

def overlay_artifact_paths(video_path: Path) -> dict[str, Path]:
    slug = slugify_video_name(video_path.name)
    stem = f"{slug}_overlay_{FAST_PROFILE['label']}"
    return {
        "stem": stem,
        "video": OUT_DIR / f"{stem}.mp4",
        "json": OUT_DIR / f"{stem}.json",
    }

def format_duration(seconds: float | None) -> str:
    if seconds is None or seconds < 0:
        return "--:--"
    total_seconds = int(round(seconds))
    minutes, secs = divmod(total_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours:d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"

def overall_fraction(update: dict[str, object]) -> float:
    stage = str(update.get("stage", ""))
    current = max(int(update.get("current", 0)), 0)
    total = max(int(update.get("total", 1)), 1)
    if stage == "done":
        return 1.0
    offset, weight = PROGRESS_STAGE_WEIGHTS.get(stage, (0.0, 0.0))
    return min(offset + (weight * (current / total)), 0.999)

selector = widgets.Dropdown(
    options=[(display_label(video), str(video)) for video in VIDEO_OPTIONS],
    value=str(DEFAULT_VIDEO),
    description="Video:",
    layout=widgets.Layout(width="520px"),
)
force_rerun = widgets.Checkbox(value=False, description="Force rerun")
render_button = widgets.Button(description="Render full overlay video", button_style="primary")
status_html = widgets.HTML()
progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Render:",
    bar_style="",
    layout=widgets.Layout(width="100%"),
)
progress_meta = widgets.HTML()
summary_html = widgets.HTML()
video_html = widgets.HTML()
render_state = {"thread": None}

def set_status(message: str, *, fraction: float, started_at: float) -> None:
    fraction = max(0.0, min(float(fraction), 1.0))
    elapsed = perf_counter() - started_at
    eta = None if fraction <= 0.01 or fraction >= 1.0 else (elapsed * (1.0 - fraction) / fraction)
    percent = int(round(fraction * 100.0))
    progress_bar.value = percent
    progress_bar.bar_style = "success" if percent >= 100 else "info"
    status_html.value = (
        "<div style='font-family:monospace; margin:8px 0 6px 0;'>"
        f"<strong>{message}</strong>"
        "</div>"
    )
    progress_meta.value = (
        "<div style='font-family:monospace; margin:6px 0 10px 0;'>"
        f"Progress: {percent}% | Elapsed: {format_duration(elapsed)} | ETA: {format_duration(eta)}"
        "</div>"
    )

def set_controls_running(is_running: bool) -> None:
    selector.disabled = is_running
    force_rerun.disabled = is_running
    render_button.disabled = is_running
    render_button.description = "Rendering..." if is_running else "Render full overlay video"

def video_widget_markup(video_path: Path) -> str:
    relative_path = video_path.relative_to(ROOT).as_posix()
    cache_buster = int(video_path.stat().st_mtime) if video_path.exists() else 0
    src = f"{quote(relative_path, safe='/')}?v={cache_buster}"
    return (
        "<video controls playsinline style='width:960px; max-width:100%; height:auto; background:#000; border:1px solid #d1d5db; border-radius:10px;'>"
        f"<source src='{src}' type='video/mp4'>"
        "</video>"
    )

def load_overlay_summary(video_path: Path, overlay_video: Path, overlay_json: Path, *, render_elapsed: float, cache_used: bool, device: str) -> None:
    data = json.loads(overlay_json.read_text())
    fps = float(data["fps"])
    frame_count = len(data["frames"])
    ball_detections = sum(frame_data.get("ball", {}).get("image_xy") is not None for frame_data in data["frames"])
    summary_html.value = (
        "<pre style='font-family:monospace; font-size:13px; line-height:1.45; margin:8px 0 12px 0;'>"
        + json.dumps(
            {
                "selected_video": video_path.name,
                "profile": FAST_PROFILE["label"],
                "device": device,
                "frames": frame_count,
                "duration_sec": round(frame_count / fps, 2) if fps else 0.0,
                "ball_detections": ball_detections,
                "players_in_first_frame": len(data["frames"][0].get("players", [])) if data["frames"] else 0,
                "artifact_video": str(overlay_video.relative_to(ROOT)),
                "artifact_json": str(overlay_json.relative_to(ROOT)),
                "render_elapsed_sec": round(render_elapsed, 2),
                "cache_used": cache_used,
            },
            indent=2,
        )
        + "</pre>"
    )
    video_html.value = video_widget_markup(overlay_video)

def show_cached_preview(video_path: Path) -> None:
    artifacts = overlay_artifact_paths(video_path)
    if artifacts["video"].exists() and artifacts["json"].exists():
        load_overlay_summary(
            video_path,
            artifacts["video"],
            artifacts["json"],
            render_elapsed=0.0,
            cache_used=True,
            device=runtime.torch_device,
        )
        set_status("Ready to play cached overlay", fraction=1.0, started_at=perf_counter())
        return
    summary_html.value = (
        "<div style='font-family:monospace; margin:8px 0 12px 0;'>"
        f"No cached overlay yet for <strong>{video_path.name}</strong>. Click <strong>Render full overlay video</strong>."
        "</div>"
    )
    video_html.value = ""
    progress_bar.value = 0
    progress_bar.bar_style = ""
    progress_meta.value = ""
    status_html.value = (
        "<div style='font-family:monospace; margin:8px 0 6px 0;'>"
        f"<strong>Ready on {runtime.torch_device}</strong>"
        "</div>"
    )

def render_overlay(video_path: Path, *, force: bool = False) -> None:
    current_runtime = detect_runtime(require_cuda=bool(PIPELINE_PROFILE.get("require_cuda", False)))
    artifacts = overlay_artifact_paths(video_path)
    overlay_video = artifacts["video"]
    overlay_json = artifacts["json"]
    started_at = perf_counter()

    def progress_handler(update: dict[str, object]) -> None:
        stage = str(update.get("stage", "working"))
        current = int(update.get("current", 0))
        total = int(update.get("total", 1))
        message = str(update.get("message", stage)).strip()
        set_status(f"{message} ({current}/{max(total, 1)})", fraction=overall_fraction(update), started_at=started_at)

    needs_rerun = force or not overlay_video.exists() or not overlay_json.exists()
    if needs_rerun:
        set_status("Starting overlay render", fraction=0.01, started_at=started_at)
        results = run_pipeline(
            video_path=video_path,
            output_dir=OUT_DIR,
            step="all",
            weights_path=WEIGHTS_PATH,
            output_stem=artifacts["stem"],
            write_video=True,
            progress_callback=progress_handler,
            **PIPELINE_PROFILE,
        )
        overlay_video = Path(results["video"])
        overlay_json = Path(results["json"])
    else:
        set_status("Using cached overlay artifacts", fraction=1.0, started_at=started_at)

    render_elapsed = perf_counter() - started_at
    load_overlay_summary(
        video_path,
        overlay_video,
        overlay_json,
        render_elapsed=render_elapsed,
        cache_used=not needs_rerun,
        device=current_runtime.torch_device,
    )
    set_status("Overlay ready", fraction=1.0, started_at=started_at)


In [2]:
def on_render_clicked(_: widgets.Button) -> None:
    thread = render_state.get("thread")
    if isinstance(thread, threading.Thread) and thread.is_alive():
        status_html.value = (
            "<div style='font-family:monospace; color:#92400e; margin:8px 0;'>"
            "A render is already running. Wait for it to finish before starting another one."
            "</div>"
        )
        return

    summary_html.value = ""
    video_html.value = ""
    started_at = perf_counter()
    set_controls_running(True)
    set_status(f"Preparing {FAST_PROFILE['label']} render on {runtime.torch_device}", fraction=0.01, started_at=started_at)

    def worker() -> None:
        try:
            render_overlay(Path(selector.value), force=force_rerun.value)
        except Exception as exc:
            progress_bar.bar_style = "danger"
            status_html.value = (
                "<div style='font-family:monospace; color:#b91c1c; margin:8px 0;'>"
                f"Overlay render failed: {exc}"
                "</div>"
            )
            progress_meta.value = (
                "<pre style='font-family:monospace; color:#7f1d1d; margin:6px 0 10px 0;'>"
                + traceback.format_exc()
                + "</pre>"
            )
        finally:
            render_state["thread"] = None
            set_controls_running(False)

    thread = threading.Thread(target=worker, daemon=True)
    render_state["thread"] = thread
    thread.start()

def on_video_changed(change: dict[str, object]) -> None:
    if change.get("name") != "value" or change.get("new") == change.get("old"):
        return
    thread = render_state.get("thread")
    if isinstance(thread, threading.Thread) and thread.is_alive():
        return
    show_cached_preview(Path(str(change["new"])))

render_button.on_click(on_render_clicked)
selector.observe(on_video_changed, names="value")
controls = widgets.HBox([selector, force_rerun, render_button])
display(widgets.VBox([controls, status_html, progress_bar, progress_meta, summary_html, video_html]))
show_cached_preview(Path(selector.value))
